# ⚽ AI/ML Football CV Analysis Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUDITaidsml/FootballCV/blob/main/football_analysis.ipynb)
[![Streamlit App](https://static.streamlit.io/badges/streamlit_badge_black_white.svg)](https://footballcv-ms.streamlit.app/)

## 📌 Project Overview
This notebook implements an end-to-end computer vision and machine learning pipeline for professional football (soccer) match analysis.

### Key Modules:
1. **Object Detection & Tracking**: YOLOv8 + ByteTrack for players, referees, and ball tracking.
2. **Camera Movement Estimation**: Lucas-Kanade Optical Flow to measure camera pan & zoom.
3. **Perspective Transformation**: Homography transformation mapping pixel coordinates to pitch meter coordinates.
4. **Ball Interpolation**: Cubic spline interpolation for missing ball detections across frames.
5. **Team Assignment**: K-Means clustering on jersey color histograms.
6. **Ball Possession**: Dynamic spatial proximity & nearest-player fallback for team possession metrics.
7. **Speed & Distance Estimator**: Frame-by-frame velocity and total distance covered per player.
8. **Video Rendering & Export**: In-place annotated MP4 output generation.

## 1. Google Colab Environment Setup

In [ ]:
# Clone project repository
!git clone https://github.com/MUDITaidsml/FootballCV.git
%cd FootballCV

# Install Linux system dependencies for OpenCV
!apt-get update -qq && !apt-get install -y -qq libgl1 libglib2.0-dev libsm6 libice6 libxext6 libxrender1

# Install Python package requirements
!pip install -q ultralytics opencv-python-headless scikit-learn pandas numpy matplotlib filterpy lapx supervision imageio imageio-ffmpeg

## 2. Import Libraries & Verify GPU

In [ ]:
import os
import sys
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import supervision as sv

# Set Ultralytics config dir to writable location
os.environ['YOLO_CONFIG_DIR'] = '/tmp/Ultralytics'

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 3. Load Project Architecture Modules

In [ ]:
from trackers import Tracker
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
from camera_movement_estimator import CameraMovementEstimator
from view_transformer import ViewTransformer
from speed_and_distance_estimator import SpeedAndDistance_Estimator
from utils.video_utils import read_video, save_video_mp4, downscale_frame

## 4. Video Frame Loading & Downscaling

In [ ]:
input_video_path = 'input_videos/sample_match.mp4'

# Create input directory if needed
os.makedirs('input_videos', exist_ok=True)

# Read video frames (up to 150 frames for fast analysis)
video_frames = read_video(input_video_path)
print(f"Successfully loaded {len(video_frames)} frames.")

# Display first frame
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(video_frames[0], cv2.COLOR_BGR2RGB))
plt.title('Frame 1 Sample')
plt.axis('off')
plt.show()

## 5. Object Detection & ByteTrack Tracking (Step 1-2)

In [ ]:
# Initialize Tracker with YOLO model
model_name = 'yolov8n.pt'  # Options: 'yolov8n.pt', 'yolov8s.pt', 'yolov8x.pt'
tracker = Tracker(model_name)

# Run mini-batch streaming detection and tracking
tracks = tracker.get_object_tracks(video_frames, read_from_stub=False)

print(f"Tracked {len(tracks.get('players', []))} frames.")
print(f"Frame 0 Player Detections: {len(tracks['players'][0])}")

## 6. Camera Movement Estimation (Step 3-4)

In [ ]:
camera_estimator = CameraMovementEstimator(video_frames[0])
camera_movement_per_frame = camera_estimator.get_camera_movement(video_frames)

# Adjust player positions for camera panning/zooming
tracker.add_position_to_tracks(tracks)
camera_estimator.add_adjust_positions_to_tracks(tracks, camera_movement_per_frame)

# Transform view to 2D pitch ground coordinates (Homography)
view_transformer = ViewTransformer()
view_transformer.add_transformed_position_to_tracks(tracks)

print("Camera movement and pitch coordinates computed.")

## 7. Ball Interpolation (Step 5)

In [ ]:
if tracks.get('ball') and any(tracks['ball']):
    tracks['ball'] = tracker.interpolate_ball_positions(tracks['ball'])
    print("Ball positions interpolated successfully.")
else:
    print("No ball detected in clip.")

## 8. Team Assignment & Jersey Clustering (Step 7)

In [ ]:
team_assigner = TeamAssigner()

# Fit K-Means clustering on the first valid player frame
for f_idx, p_dict in enumerate(tracks.get('players', [])):
    if len(p_dict) >= 2:
        team_assigner.assign_team_color(video_frames[f_idx], p_dict)
        break

# Assign teams to all tracked players
for frame_num, player_track in enumerate(tracks.get('players', [])):
    for player_id, track in player_track.items():
        team = team_assigner.get_player_team(video_frames[frame_num], track['bbox'], player_id)
        tracks['players'][frame_num][player_id]['team'] = team
        tracks['players'][frame_num][player_id]['team_color'] = team_assigner.team_colors.get(team, (0, 0, 255))

print(f"Team Colors: Team 1 = {team_assigner.team_colors.get(1)}, Team 2 = {team_assigner.team_colors.get(2)}")

## 9. Ball Possession & Speed Metrics (Step 6 & 8)

In [ ]:
# Speed & Distance Estimation
speed_and_distance_estimator = SpeedAndDistance_Estimator()
speed_and_distance_estimator.add_speed_and_distance_to_tracks(tracks)

# Ball Possession Assignment
player_assigner = PlayerBallAssigner()
team_ball_control = []
for frame_num, player_track in enumerate(tracks.get('players', [])):
    ball_entry = tracks['ball'][frame_num] if frame_num < len(tracks['ball']) else {}
    ball_bbox = ball_entry.get(1, {}).get('bbox')
    if ball_bbox is None:
        team_ball_control.append(team_ball_control[-1] if team_ball_control else 1)
        continue
    assigned_player = player_assigner.assign_ball_to_player(player_track, ball_bbox)
    if assigned_player != -1 and assigned_player in player_track:
        tracks['players'][frame_num][assigned_player]['has_ball'] = True
        team_ball_control.append(tracks['players'][frame_num][assigned_player].get('team', 1))
    else:
        team_ball_control.append(team_ball_control[-1] if team_ball_control else 1)

team_ball_control = np.array(team_ball_control)
t1_pct = (team_ball_control == 1).mean() * 100
t2_pct = (team_ball_control == 2).mean() * 100
print(f"Possession: Team 1 = {t1_pct:.1f}%, Team 2 = {t2_pct:.1f}%")

## 10. Dashboard Analytics & Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart for Ball Possession
axes[0].pie([t1_pct, t2_pct], labels=['Team 1', 'Team 2'], colors=['#3b82f6', '#ef4444'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Ball Possession Distribution')

# Plot Camera Movement X & Y
cam_x = [m[0] for m in camera_movement_per_frame]
cam_y = [m[1] for m in camera_movement_per_frame]
axes[1].plot(cam_x, label='Camera X (Pan)', color='#10b981')
axes[1].plot(cam_y, label='Camera Y (Tilt)', color='#f59e0b')
axes[1].set_xlabel('Frame Index')
axes[1].set_ylabel('Pixel Offset')
axes[1].set_title('Camera Movement Tracking')
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Render Annotated Video Output (Step 9)

In [ ]:
# Draw visual annotations on video frames
video_frames = tracker.draw_annotations(video_frames, tracks, team_ball_control)
video_frames = camera_estimator.draw_camera_movement(video_frames, camera_movement_per_frame)
speed_and_distance_estimator.draw_speed_and_distance(video_frames, tracks)

# Save output video file
os.makedirs('output_videos', exist_ok=True)
output_video_path = 'output_videos/analyzed_output.mp4'
save_video_mp4(video_frames, output_video_path)
print(f"Annotated video saved successfully to {output_video_path}")